# 05 — Export for Tableau Dashboard
**Goal:** Menyiapkan semua CSV yang dibutuhkan Tableau dalam format yang clean, ringan, dan siap connect.

## Data Sources yang akan dibuat:
| File | Digunakan untuk |
|---|---|
| `tableau_transactions.csv` | Main fact table — overview & drill-down |
| `tableau_fraud_summary.csv` | KPI cards & trend chart |
| `tableau_risk_heatmap.csv` | Heatmap fraud by method & provider |
| `tableau_network_top.csv` | Network/scatter plot buyer-seller |
| `tableau_monthly_trend.csv` | Time series chart |

## 0. Setup

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

PROCESSED_DIR = Path('..') / 'data' / 'processed'
TABLEAU_DIR   = Path('..') / 'tableau'
TABLEAU_DIR.mkdir(exist_ok=True)

print('Tableau output dir:', TABLEAU_DIR.resolve())

Tableau output dir: D:\Project\data-engineer\paperid-fraud-detection\tableau


## 1. Load Semua Data

In [2]:
# Load dari src untuk dapat kolom lengkap
from src.data.loader import load_all
from src.data.cleaner import clean_all

raw         = load_all()
raw         = clean_all(raw)
company     = raw['company']
promotion   = raw['promotion']

# Load hasil modeling
pred_df = pd.read_csv(PROCESSED_DIR / 'fraud_predictions.csv')
pred_df['transaction_created_datetime'] = pd.to_datetime(pred_df['transaction_created_datetime'])

# Load network
nodes_df = pd.read_csv(PROCESSED_DIR / 'network_nodes.csv')
edges_df = pd.read_csv(PROCESSED_DIR / 'network_edges.csv')

print(f'pred_df  : {pred_df.shape}')
print(f'nodes_df : {nodes_df.shape}')
print(f'edges_df : {edges_df.shape}')

pred_df  : (26772, 27)
nodes_df : (6843, 8)
edges_df : (5000, 7)


## 2. tableau_transactions.csv — Main Fact Table

In [3]:
# Join dengan company untuk dapat nama/tipe, dan promotion untuk nama promo
company_map = company[['company_id', 'company_type_group',
                        'company_kyc_status_name', 'company_age_days']].drop_duplicates('company_id')

promo_map = promotion[['dpt_promotion_id', 'promotion_name']].drop_duplicates('dpt_promotion_id') \
    if 'dpt_promotion_id' in promotion.columns else None

tx = pred_df[[
    'dpt_id', 'buyer_id', 'seller_id',
    'transaction_amount', 'transaction_created_datetime',
    'is_fraud', 'fraud_probability', 'fraud_predicted', 'risk_level',
    'is_outlier_iqr', 'anomaly', 'transaction_count',
    'is_burst', 'promo_usage_count_buyer',
]].copy()

# Tambah kolom waktu untuk Tableau filter
tx['year']    = tx['transaction_created_datetime'].dt.year
tx['month']   = tx['transaction_created_datetime'].dt.month
tx['month_name'] = tx['transaction_created_datetime'].dt.strftime('%b %Y')
tx['day_of_week'] = tx['transaction_created_datetime'].dt.day_name()
tx['hour']    = tx['transaction_created_datetime'].dt.hour

# Join buyer company info
tx = tx.merge(
    company_map.rename(columns={
        'company_id': 'buyer_id',
        'company_type_group': 'buyer_type',
        'company_kyc_status_name': 'buyer_kyc',
        'company_age_days': 'buyer_age_days'
    }),
    on='buyer_id', how='left'
)

# Join seller company info
tx = tx.merge(
    company_map.rename(columns={
        'company_id': 'seller_id',
        'company_type_group': 'seller_type',
        'company_kyc_status_name': 'seller_kyc',
        'company_age_days': 'seller_age_days'
    }),
    on='seller_id', how='left'
)

# Risk label
tx['risk_level'] = tx['risk_level'].astype(str)
tx['fraud_label'] = tx['is_fraud'].map({0: 'Normal', 1: 'Fraud'})

tx.to_csv(TABLEAU_DIR / 'tableau_transactions.csv', index=False)
print(f'tableau_transactions.csv : {tx.shape}')
print(tx.dtypes)

tableau_transactions.csv : (26772, 26)
dpt_id                                  object
buyer_id                                object
seller_id                               object
transaction_amount                     float64
transaction_created_datetime    datetime64[ns]
is_fraud                                 int64
fraud_probability                      float64
fraud_predicted                          int64
risk_level                              object
is_outlier_iqr                           int64
anomaly                                  int64
transaction_count                        int64
is_burst                                 int64
promo_usage_count_buyer                  int64
year                                     int32
month                                    int32
month_name                              object
day_of_week                             object
hour                                     int32
buyer_type                              object
buyer_kyc            

## 3. tableau_fraud_summary.csv — KPI & Overview

In [4]:
summary = pd.DataFrame([{
    'total_transactions'    : len(tx),
    'total_amount'          : tx['transaction_amount'].sum(),
    'fraud_transactions'    : tx['is_fraud'].sum(),
    'fraud_amount'          : tx[tx['is_fraud'] == 1]['transaction_amount'].sum(),
    'fraud_rate_pct'        : tx['is_fraud'].mean() * 100,
    'avg_fraud_probability' : tx['fraud_probability'].mean(),
    'high_risk_count'       : (tx['risk_level'] == 'High').sum(),
    'medium_risk_count'     : (tx['risk_level'] == 'Medium').sum(),
    'low_risk_count'        : (tx['risk_level'] == 'Low').sum(),
    'burst_tx_count'        : tx['is_burst'].sum(),
    'unique_buyers'         : tx['buyer_id'].nunique(),
    'unique_sellers'        : tx['seller_id'].nunique(),
}])

summary.to_csv(TABLEAU_DIR / 'tableau_fraud_summary.csv', index=False)
print('tableau_fraud_summary.csv:')
print(summary.T.to_string())

tableau_fraud_summary.csv:
                                  0
total_transactions     2.677200e+04
total_amount           8.203013e+11
fraud_transactions     4.633000e+03
fraud_amount           2.670995e+11
fraud_rate_pct         1.730539e+01
avg_fraud_probability  2.821244e-01
high_risk_count        4.341000e+03
medium_risk_count      3.636000e+03
low_risk_count         1.879500e+04
burst_tx_count         8.763000e+03
unique_buyers          6.498000e+03
unique_sellers         4.130000e+02


## 4. tableau_monthly_trend.csv — Time Series

In [5]:
monthly = (
    tx.groupby(['year', 'month', 'month_name'])
    .agg(
        total_transactions = ('dpt_id', 'count'),
        total_amount       = ('transaction_amount', 'sum'),
        fraud_count        = ('is_fraud', 'sum'),
        fraud_amount       = ('transaction_amount', lambda x: x[tx.loc[x.index, 'is_fraud'] == 1].sum()),
        avg_fraud_prob     = ('fraud_probability', 'mean'),
        high_risk_count    = ('risk_level', lambda x: (x == 'High').sum()),
    )
    .reset_index()
)
monthly['fraud_rate_pct'] = monthly['fraud_count'] / monthly['total_transactions'] * 100
monthly['sort_key']       = monthly['year'].astype(str) + monthly['month'].astype(str).str.zfill(2)
monthly = monthly.sort_values('sort_key').drop(columns='sort_key')

monthly.to_csv(TABLEAU_DIR / 'tableau_monthly_trend.csv', index=False)
print(f'tableau_monthly_trend.csv : {monthly.shape}')
print(monthly[['month_name', 'total_transactions', 'fraud_count', 'fraud_rate_pct']].to_string(index=False))

tableau_monthly_trend.csv : (18, 10)
month_name  total_transactions  fraud_count  fraud_rate_pct
  Feb 2022                   1            0        0.000000
  Jun 2022                   1            0        0.000000
  Sep 2022                   2            0        0.000000
  Oct 2022                   3            0        0.000000
  Nov 2022                   3            0        0.000000
  Dec 2022                  26            4       15.384615
  Jan 2023                 850          167       19.647059
  Feb 2023                1167          299       25.621251
  Mar 2023                1886          477       25.291622
  Apr 2023                1689          365       21.610420
  May 2023                2058          445       21.622935
  Jun 2023                2557          609       23.816973
  Jul 2023                3133          718       22.917332
  Aug 2023                2319          348       15.006468
  Sep 2023                2308          284       12.305026
  O

## 5. tableau_risk_heatmap.csv — Fraud by Buyer Type & KYC

In [6]:
heatmap = (
    tx.groupby(['buyer_type', 'buyer_kyc'])
    .agg(
        total_tx    = ('dpt_id', 'count'),
        fraud_count = ('is_fraud', 'sum'),
        total_amount= ('transaction_amount', 'sum'),
        avg_fraud_prob = ('fraud_probability', 'mean'),
    )
    .reset_index()
)
heatmap['fraud_rate_pct'] = heatmap['fraud_count'] / heatmap['total_tx'] * 100
heatmap = heatmap.fillna('Unknown')

heatmap.to_csv(TABLEAU_DIR / 'tableau_risk_heatmap.csv', index=False)
print(f'tableau_risk_heatmap.csv : {heatmap.shape}')
print(heatmap.sort_values('fraud_rate_pct', ascending=False).head(10).to_string(index=False))

tableau_risk_heatmap.csv : (23, 7)
buyer_type              buyer_kyc  total_tx  fraud_count  total_amount  avg_fraud_prob  fraud_rate_pct
        CV         AKUN_DIBEKUKAN       275          275  7.673275e+09        0.852660      100.000000
        PT         AKUN_DIBEKUKAN       281          281  1.729924e+10        0.695480      100.000000
PERORANGAN         AKUN_DIBEKUKAN      3968         3961  2.372413e+11        0.834304       99.823589
        CV DOKUMEN_KURANG_LENGKAP        10            1  7.156079e+08        0.286968       10.000000
        PT         BELUM_VALIDASI       201           10  9.253425e+08        0.113616        4.975124
        PT DOKUMEN_KURANG_LENGKAP        40            1  7.618252e+08        0.097222        2.500000
PERORANGAN DOKUMEN_KURANG_LENGKAP       385            8  2.542202e+09        0.121212        2.077922
PERORANGAN                  DRAFT        89            1  5.989114e+07        0.100896        1.123596
PERORANGAN      VALIDASI_BERHASIL     

## 6. tableau_network_top.csv — Scatter/Network Plot

In [7]:
# Top 500 nodes by pagerank untuk scatter plot di Tableau
network_top = nodes_df.sort_values('pagerank', ascending=False).head(500).copy()
network_top['node_short'] = network_top['node'].str[:12]
network_top['risk_label'] = network_top['avg_fraud_prob'].apply(
    lambda x: 'High' if x > 0.6 else ('Medium' if x > 0.3 else 'Low')
)

network_top.to_csv(TABLEAU_DIR / 'tableau_network_top.csv', index=False)
print(f'tableau_network_top.csv : {network_top.shape}')
print(network_top[['node_short','pagerank','fraud_count','avg_fraud_prob','risk_label']].head(10).to_string(index=False))

tableau_network_top.csv : (500, 10)
  node_short  pagerank  fraud_count  avg_fraud_prob risk_label
5d2233f5a1a6  0.430581         4549        0.285786        Low
93628e8e3030  0.000635            0        0.084404        Low
cc5931492ffb  0.000490            0        0.061424        Low
388dbd8594db  0.000451            0        0.066214        Low
985cafd6d0d1  0.000433            0        0.059216        Low
6e0fe0280311  0.000398            0        0.068851        Low
ead119c5d113  0.000391           13        0.865647       High
f8f0c08ca683  0.000366           12        0.878849       High
266acfce09ab  0.000246            0        0.047033        Low
cc35d6ec9ad1  0.000207            0        0.057541        Low


## 7. Verifikasi Semua File

In [8]:
print('=== FILE SIAP UNTUK TABLEAU ===')
print(f'{"File":<40} {"Rows":>8} {"Size":>10}')
print('-' * 60)
for f in sorted(TABLEAU_DIR.iterdir()):
    tmp = pd.read_csv(f)
    size_kb = f.stat().st_size / 1024
    print(f'{f.name:<40} {len(tmp):>8,} {size_kb:>8.1f} KB')

print('\n=== PANDUAN CONNECT KE TABLEAU ===')
print('1. Buka Tableau Desktop / Tableau Public')
print('2. Connect > Text File > pilih tableau_transactions.csv sebagai primary')
print('3. Drag tableau_monthly_trend.csv → join on year + month')
print('4. Drag tableau_risk_heatmap.csv  → join on buyer_type + buyer_kyc')
print('5. Buat sheet terpisah untuk tableau_network_top.csv')
print('\nRecommended dashboard sheets:')
print('  Sheet 1 : Overview KPI (dari fraud_summary + transactions)')
print('  Sheet 2 : Trend Bulanan (dari monthly_trend)')
print('  Sheet 3 : Risk Heatmap by Company Type (dari risk_heatmap)')
print('  Sheet 4 : Network Scatter — Pagerank vs Fraud Prob (dari network_top)')
print('  Sheet 5 : Transaction Detail Table (dari transactions dengan filter risk_level)')

=== FILE SIAP UNTUK TABLEAU ===
File                                         Rows       Size
------------------------------------------------------------
tableau_fraud_summary.csv                       1      0.3 KB
tableau_monthly_trend.csv                      18      1.7 KB
tableau_network_top.csv                       500     63.3 KB
tableau_risk_heatmap.csv                       23      1.7 KB
tableau_transactions.csv                   26,772   9252.3 KB

=== PANDUAN CONNECT KE TABLEAU ===
1. Buka Tableau Desktop / Tableau Public
2. Connect > Text File > pilih tableau_transactions.csv sebagai primary
3. Drag tableau_monthly_trend.csv → join on year + month
4. Drag tableau_risk_heatmap.csv  → join on buyer_type + buyer_kyc
5. Buat sheet terpisah untuk tableau_network_top.csv

Recommended dashboard sheets:
  Sheet 1 : Overview KPI (dari fraud_summary + transactions)
  Sheet 2 : Trend Bulanan (dari monthly_trend)
  Sheet 3 : Risk Heatmap by Company Type (dari risk_heatmap)
  Sheet 4 